In [ ]:
import json
import sqlite3
import pandas as pd

DB = "../backtest_results.db"

con = sqlite3.connect(DB)

# ── Runs ──────────────────────────────────────────────────────────────────────
runs = pd.read_sql("SELECT run_id, strategy, run_time, summary FROM backtest_runs ORDER BY run_id DESC", con)
runs["summary"] = runs["summary"].apply(json.loads)
runs = pd.concat([runs.drop(columns="summary"), runs["summary"].apply(pd.Series)], axis=1)
display(runs)

# ── Pick a run ────────────────────────────────────────────────────────────────
RUN_ID = int(runs["run_id"].iloc[0])   # change to inspect a different run
print(f"\nLoading run_id={RUN_ID}  ({runs.loc[runs.run_id==RUN_ID, 'strategy'].iloc[0]})")

# ── Equity curve ──────────────────────────────────────────────────────────────
equity = pd.read_sql(
    "SELECT timestamp, equity FROM backtest_equity WHERE run_id=? ORDER BY id",
    con, params=(RUN_ID,), parse_dates=["timestamp"]
)
equity.plot(x="timestamp", y="equity", figsize=(14, 4), title=f"Equity curve — run {RUN_ID}", legend=False)

# ── Trades ────────────────────────────────────────────────────────────────────
trades = pd.read_sql(
    "SELECT instrument_key, direction, entry_time, entry_price, exit_time, exit_price, quantity, pnl "
    "FROM backtest_trades WHERE run_id=? ORDER BY entry_time",
    con, params=(RUN_ID,)
)
trades["instrument"] = trades["instrument_key"].str.split("|").str[-1]
display(trades.drop(columns="instrument_key"))

con.close()